## Introduction

This notebook is a **universal Braket SV1 task resumer/decoder**.

It does **not** call `task.result()` (which can hang notebooks, hit signature expiry, or event-loop issues).
Instead, it:
1) lists `results.json` under a Braket S3 prefix
2) extracts measurement samples safely
3) decodes best bitstrings (optionally using a provided QUBO matrix)
4) writes a “best result” JSON plus optional “selected trials” CSV

You can reuse it for Scenario B/C (and future scenarios) by setting:
- `BRAKET_BUCKET` and `BRAKET_PREFIX`
- optional `QUBO_JSON_PATH` to compute energies
- optional `NCT_IDS_PATH` (or parse from QUBO JSON)

Outputs:
- `data/results/10d_best_from_s3.json`
- `data/results/10d_decoded_rows.csv`


In [1]:
# ============================================================
# Cell 1 — Setup: imports, config, and paths
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import boto3

RESULTS_DIR = Path("data/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- REQUIRED: Braket output location (amazon-braket-* bucket) ---
BRAKET_BUCKET = "amazon-braket-us-west-2-581610642254"

# Point to the scenario prefix you want to decode.
# Example for scenario_C:
# BRAKET_PREFIX = "clinical-trials-data/results/sv1_tasks/scenario_C/"
BRAKET_PREFIX = "clinical-trials-data/results/sv1_tasks/scenario_C/"

# Optional: compute energies if you provide a QUBO JSON artifact
QUBO_JSON_PATH = Path("data/qubo_scenarios/scenario_C_qubo.json")  # set to your scenario
# Optional: if you have a standalone list of ids, set it; otherwise parsed from QUBO if possible
NCT_IDS_PATH = None  # e.g., Path("data/scenarios/scenario_C_nct_ids.csv")

# Outputs
OUT_ROWS = RESULTS_DIR / "10d_decoded_rows.csv"
OUT_BEST = RESULTS_DIR / "10d_best_from_s3.json"

# AWS client
s3 = boto3.client("s3", region_name="us-west-2")

print("Config:")
print("  bucket:", BRAKET_BUCKET)
print("  prefix:", BRAKET_PREFIX)
print("  qubo  :", QUBO_JSON_PATH if QUBO_JSON_PATH.exists() else "(none)")


Config:
  bucket: amazon-braket-us-west-2-581610642254
  prefix: clinical-trials-data/results/sv1_tasks/scenario_C/
  qubo  : data/qubo_scenarios/scenario_C_qubo.json


### What Cell 1 Just Did

- Configured the Braket S3 location (bucket + prefix).
- Set optional QUBO inputs so we can compute energies if available.
- Set output paths for decoded rows and the “best” artifact.


In [2]:
# ============================================================
# Cell 2 — Helpers: list results.json + parse measurement samples
# ============================================================

def list_results_json(bucket: str, prefix: str) -> list[str]:
    keys = []
    token = None
    while True:
        kwargs = {"Bucket": bucket, "Prefix": prefix}
        if token:
            kwargs["ContinuationToken"] = token
        resp = s3.list_objects_v2(**kwargs)

        for it in resp.get("Contents", []):
            k = it["Key"]
            if k.endswith("results.json"):
                keys.append(k)

        if resp.get("IsTruncated"):
            token = resp.get("NextContinuationToken")
        else:
            break
    return sorted(keys)

def load_json_s3(bucket: str, key: str) -> dict:
    raw = s3.get_object(Bucket=bucket, Key=key)["Body"].read().decode("utf-8")
    return json.loads(raw)

def extract_samples(obj: dict) -> list[str]:
    """
    Supports common Braket results.json shapes:
      - measurementCounts: dict bitstring->count
      - measurements: list of arrays (0/1) or bitstring-like entries
    """
    # counts-style (best for speed)
    if isinstance(obj.get("measurementCounts"), dict):
        out = []
        for k, v in obj["measurementCounts"].items():
            out.extend([str(k).replace(" ", "")] * int(v))
        return out

    # shot-by-shot measurements
    m = obj.get("measurements")
    if isinstance(m, list) and m:
        if isinstance(m[0], (list, tuple, np.ndarray)):
            return ["".join(str(int(b)) for b in row) for row in m]
        if isinstance(m[0], str):
            return [str(x).replace(" ", "") for x in m]

    return []

keys = list_results_json(BRAKET_BUCKET, BRAKET_PREFIX)
print(f"Found {len(keys)} results.json file(s).")
print("First few:")
for k in keys[:5]:
    print(" -", k)


Found 7 results.json file(s).
First few:
 - clinical-trials-data/results/sv1_tasks/scenario_C/04b33e82-35e9-418f-872b-25a7012f6bac/results.json
 - clinical-trials-data/results/sv1_tasks/scenario_C/24a0561c-0c36-45f9-ad73-a76d50cb67cb/results.json
 - clinical-trials-data/results/sv1_tasks/scenario_C/2e855d38-5da3-459e-a800-7bbc35148b38/results.json
 - clinical-trials-data/results/sv1_tasks/scenario_C/435154ab-827e-460a-b503-06b1662e7968/results.json
 - clinical-trials-data/results/sv1_tasks/scenario_C/6b9614ad-e00d-4331-97d1-0961d1aea55b/results.json


### What Cell 2 Just Did

- Listed all `results.json` objects under your scenario prefix.
- Added robust parsing so we can decode either:
  - count dictionaries (`measurementCounts`) or
  - explicit shot measurements (`measurements`).


In [3]:
# ============================================================
# Cell 3 — load QUBO and ID mapping for energy scoring
# ============================================================

def parse_qubo_to_matrix(payload: dict):
    # unwrap single-key wrapper
    if isinstance(payload, dict) and len(payload) == 1:
        only_key = next(iter(payload.keys()))
        if isinstance(payload[only_key], dict):
            payload = payload[only_key]

    if not isinstance(payload, dict):
        return None, None

    # ids
    nct_ids = None
    for key in ["nct_ids", "trial_ids", "ids", "variables"]:
        if key in payload and isinstance(payload[key], list):
            nct_ids = [str(x) for x in payload[key]]
            break

    # dense
    for key in ["Q", "qubo_matrix", "matrix"]:
        if key in payload:
            Q = np.array(payload[key], dtype=float)
            if Q.ndim == 2 and Q.shape[0] == Q.shape[1]:
                if nct_ids is None:
                    nct_ids = [f"var_{i}" for i in range(Q.shape[0])]
                return Q, nct_ids

    # dict terms
    n = payload.get("n") or (len(nct_ids) if nct_ids else None)
    if n is None:
        return None, None
    n = int(n)
    Q = np.zeros((n, n), dtype=float)

    if "linear" in payload and isinstance(payload["linear"], dict):
        for i, v in payload["linear"].items():
            Q[int(i), int(i)] += float(v)

    if "quadratic" in payload and isinstance(payload["quadratic"], dict):
        for k, v in payload["quadratic"].items():
            s = str(k).strip().replace("(", "").replace(")", "").replace("[", "").replace("]", "")
            parts = [p.strip() for p in s.split(",") if p.strip()]
            if len(parts) != 2:
                continue
            i, j = int(parts[0]), int(parts[1])
            Q[i, j] += float(v)

    if "qubo" in payload and isinstance(payload["qubo"], dict):
        for k, v in payload["qubo"].items():
            s = str(k).strip().replace("(", "").replace(")", "").replace("[", "").replace("]", "")
            parts = [p.strip() for p in s.split(",") if p.strip()]
            if len(parts) != 2:
                continue
            i, j = int(parts[0]), int(parts[1])
            Q[i, j] += float(v)

    if nct_ids is None:
        nct_ids = [f"var_{i}" for i in range(n)]
    return Q, nct_ids

Q = None
nct_ids = None

if QUBO_JSON_PATH.exists():
    payload = json.loads(QUBO_JSON_PATH.read_text())
    Q, nct_ids = parse_qubo_to_matrix(payload)
    if Q is None:
        print("Warning: Could not parse QUBO into a matrix; decoding will proceed without energy scoring.")
    else:
        # symmetrize for stable scoring
        Q = 0.5 * (Q + Q.T)
        print("Loaded QUBO matrix:", Q.shape)

if NCT_IDS_PATH is not None and Path(NCT_IDS_PATH).exists():
    df_ids = pd.read_csv(NCT_IDS_PATH)
    if "nct_id" in df_ids.columns:
        nct_ids = [str(x) for x in df_ids["nct_id"].tolist()]

print("nct_ids available:", bool(nct_ids))


Loaded QUBO matrix: (60, 60)
nct_ids available: True


### What Cell 3 Just Did

- Loaded the scenario QUBO JSON and converted it into a dense Q matrix (when possible).
- Optionally loaded an external NCT ID list if you have one.
- If Q is available, we can score each decoded bitstring by QUBO energy; otherwise we still extract best-by-frequency samples.


In [6]:
# ============================================================
# Cell 4 — Decode each results.json robustly (pool-safe) and pick the best
# ============================================================

from collections import Counter

def _extract_measurement_rows(obj: dict):
    """
    Returns list of measurement rows if present.
    Expected row forms:
      - list[int] (0/1)
      - numpy arrays
    """
    m = obj.get("measurements", None)
    if not isinstance(m, list) or len(m) == 0:
        return []
    return m

def _rows_to_bitstrings(rows):
    out = []
    for row in rows:
        # row can be list/tuple/np.ndarray of 0/1
        if isinstance(row, (list, tuple, np.ndarray)):
            try:
                out.append("".join(str(int(b)) for b in row))
            except Exception:
                continue
        elif isinstance(row, str):
            s = row.replace(" ", "").strip()
            if set(s) <= {"0", "1"}:
                out.append(s)
    return out

def _best_by_frequency(bitstrings):
    if not bitstrings:
        return None
    c = Counter(bitstrings)
    s, n = c.most_common(1)[0]
    return {"bitstring": s, "support": int(n), "energy": None, "bit_order": "LR", "scoring": "frequency"}

def _qubo_energy(Qmat: np.ndarray, x01: np.ndarray) -> float:
    x = x01.astype(int)
    return float(x.T @ Qmat @ x)

def _best_by_energy_if_possible(Qmat: np.ndarray, bitstrings):
    """
    If Qmat dimension matches bitstring length, pick minimum-energy bitstring.
    Also tries reversed order and keeps the better energy.
    """
    if Qmat is None or not bitstrings:
        return None

    n = int(Qmat.shape[0])
    # keep only exact-length strings
    cand = [s for s in bitstrings if isinstance(s, str) and len(s) == n and set(s) <= {"0", "1"}]
    if not cand:
        return None

    freq = Counter(cand)

    best = None
    for s, support in freq.items():
        x_lr = np.array([int(ch) for ch in s], dtype=int)
        e_lr = _qubo_energy(Qmat, x_lr)

        x_rl = x_lr[::-1]
        e_rl = _qubo_energy(Qmat, x_rl)

        if e_rl < e_lr:
            e = float(e_rl)
            s_use = s[::-1]
            order = "RL->LR"
        else:
            e = float(e_lr)
            s_use = s
            order = "LR"

        row = {
            "bitstring": s_use,
            "support": int(support),
            "energy": e,
            "bit_order": order,
            "scoring": "qubo_energy",
        }
        if best is None or row["energy"] < best["energy"]:
            best = row

    return best

rows = []
diag_rows = []
best_global = None

for key in keys:
    obj = load_json_s3(BRAKET_BUCKET, key)

    measured_qubits = obj.get("measuredQubits", None)
    n_meas = len(measured_qubits) if isinstance(measured_qubits, list) else None

    mrows = _extract_measurement_rows(obj)
    bitstrings = _rows_to_bitstrings(mrows)

    diag_rows.append({
        "results_json_key": key,
        "n_samples": len(bitstrings),
        "measured_qubits_len": n_meas,
        "qubo_dim": (int(Q.shape[0]) if Q is not None else None),
        "top_level_keys": ",".join(list(obj.keys())[:12]),
    })

    if not bitstrings:
        continue

    # If Q exists BUT dimension mismatch, we intentionally fall back to frequency scoring.
    best = None
    if Q is not None and n_meas is not None and int(Q.shape[0]) == int(n_meas):
        best = _best_by_energy_if_possible(Q, bitstrings)

    if best is None:
        best = _best_by_frequency(bitstrings)

    best["results_json_key"] = key
    best["measured_qubits_len"] = n_meas
    best["qubo_dim_used"] = (int(Q.shape[0]) if (Q is not None and best["scoring"] == "qubo_energy") else None)

    rows.append(best)

    # Choose global best:
    # - prefer lowest energy if we have energy
    # - else prefer highest support
    if best_global is None:
        best_global = best
    else:
        if best_global["energy"] is not None and best["energy"] is not None:
            if best["energy"] < best_global["energy"]:
                best_global = best
        elif best_global["energy"] is None and best["energy"] is not None:
            best_global = best
        else:
            if best["support"] > best_global["support"]:
                best_global = best

decoded_df = pd.DataFrame(rows)
diag_df = pd.DataFrame(diag_rows)

OUT_DIAG = RESULTS_DIR / "10d_decode_diagnostics.csv"
diag_df.to_csv(OUT_DIAG, index=False)

print("Decoded rows:", len(decoded_df))
print("Wrote diagnostics:", OUT_DIAG)
display(diag_df.head(10))

if decoded_df.empty or best_global is None:
    raise RuntimeError(
        "No decodable results were found under the given prefix.\n"
        "But if diagnostics shows n_samples > 0, then measurement parsing is OK—"
        "we may be filtering due to unexpected row formats.\n"
        f"See: {OUT_DIAG}"
    )

decoded_df.to_csv(OUT_ROWS, index=False)
print("Wrote decoded rows:", OUT_ROWS)
display(decoded_df.head(10))

best_payload = {
    "source": "s3_results_json_resume",
    "braket_bucket": BRAKET_BUCKET,
    "braket_prefix": BRAKET_PREFIX,
    "results_json_key": best_global["results_json_key"],
    "bitstring": best_global["bitstring"],
    "support": int(best_global["support"]),
    "energy": best_global["energy"],  # may be None if pooled or Q missing
    "bit_order": best_global.get("bit_order", None),
    "scoring": best_global.get("scoring", None),
    "measured_qubits_len": best_global.get("measured_qubits_len", None),
    "qubo_dim_used": best_global.get("qubo_dim_used", None),
    "used_qubo_scoring": bool(best_global.get("scoring") == "qubo_energy"),
    "qubo_path": str(QUBO_JSON_PATH) if QUBO_JSON_PATH.exists() else None,
}

OUT_BEST.write_text(json.dumps(best_payload, indent=2))
print("Wrote best summary:", OUT_BEST)
print(best_payload)


Decoded rows: 7
Wrote diagnostics: data/results/10d_decode_diagnostics.csv


,results_json_key,n_samples,measured_qubits_len,qubo_dim,top_level_keys
0,clinical-trials-data/results/sv1_tasks/scenari...,100,34,60,"braketSchemaHeader,measurements,resultTypes,me..."
1,clinical-trials-data/results/sv1_tasks/scenari...,200,34,60,"braketSchemaHeader,measurements,resultTypes,me..."
2,clinical-trials-data/results/sv1_tasks/scenari...,100,34,60,"braketSchemaHeader,measurements,resultTypes,me..."
3,clinical-trials-data/results/sv1_tasks/scenari...,200,34,60,"braketSchemaHeader,measurements,resultTypes,me..."
4,clinical-trials-data/results/sv1_tasks/scenari...,100,34,60,"braketSchemaHeader,measurements,resultTypes,me..."
5,clinical-trials-data/results/sv1_tasks/scenari...,200,34,60,"braketSchemaHeader,measurements,resultTypes,me..."
6,clinical-trials-data/results/sv1_tasks/scenari...,100,34,60,"braketSchemaHeader,measurements,resultTypes,me..."


Wrote decoded rows: data/results/10d_decoded_rows.csv


,bitstring,support,energy,bit_order,scoring,results_json_key,measured_qubits_len,qubo_dim_used
0,0000000000001010000000000000000000,2,None,LR,frequency,clinical-trials-data/results/sv1_tasks/scenari...,34,None
1,0010001101000000100000101001000001,1,None,LR,frequency,clinical-trials-data/results/sv1_tasks/scenari...,34,None
2,1000000101100100110011000000001101,1,None,LR,frequency,clinical-trials-data/results/sv1_tasks/scenari...,34,None
3,1010100010011010000001111111110010,1,None,LR,frequency,clinical-trials-data/results/sv1_tasks/scenari...,34,None
4,1000000000010001010100110000001000,1,None,LR,frequency,clinical-trials-data/results/sv1_tasks/scenari...,34,None
5,0000011111111110000000011001111100,1,None,LR,frequency,clinical-trials-data/results/sv1_tasks/scenari...,34,None
6,1111111011101111001101101111111111,1,None,LR,frequency,clinical-trials-data/results/sv1_tasks/scenari...,34,None


Wrote best summary: data/results/10d_best_from_s3.json
{'source': 's3_results_json_resume', 'braket_bucket': 'amazon-braket-us-west-2-581610642254', 'braket_prefix': 'clinical-trials-data/results/sv1_tasks/scenario_C/', 'results_json_key': 'clinical-trials-data/results/sv1_tasks/scenario_C/04b33e82-35e9-418f-872b-25a7012f6bac/results.json', 'bitstring': '0000000000001010000000000000000000', 'support': 2, 'energy': None, 'bit_order': 'LR', 'scoring': 'frequency', 'measured_qubits_len': 34, 'qubo_dim_used': None, 'used_qubo_scoring': False, 'qubo_path': 'data/qubo_scenarios/scenario_C_qubo.json'}


### What Cell 4 Just Did

- Loaded every Braket `results.json` under the configured S3 prefix and extracted shot-by-shot measurements.
- Converted measurements into bitstrings and decoded a “best” bitstring per task.
- **Pool-safe behavior:** if the QUBO matrix dimension does not match the measured bitstring length (common when we pool down to SV1’s 34 qubits), the notebook automatically falls back to **best-by-frequency** instead of failing.
- Wrote:
  - `data/results/10d_decode_diagnostics.csv` (what we found per task)
  - `data/results/10d_decoded_rows.csv` (best decoded result per task)
  - `data/results/10d_best_from_s3.json` (single best overall for the prefix)



In [8]:
# ============================================================
# Cell 5 — Pool-aware energy scoring (if pool_idx is available)
# ============================================================

import ast

# OPTIONAL inputs: provide ONE of these if you have it for the scenario
# (These are produced by your SV1 notebooks/runners in earlier work.)
POOL_SOURCES = [
    Path("data/results/09c_sv1_tasks_scenario_C.csv"),
    Path("data/results/08a_sv1_tasks_scenario_B.csv"),
    Path("data/results/09c_sv1_tasks_scenario_C.csv"),
]

def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

POOL_TASKS_CSV = first_existing(POOL_SOURCES)

def load_pool_idx_from_tasks_csv(p: Path):
    """
    Expects a column that contains the pool index list.
    Common candidates: pool_idx, pool_indices.
    """
    df = pd.read_csv(p)
    col = None
    for c in ["pool_idx", "pool_indices", "pool_index"]:
        if c in df.columns:
            col = c
            break
    if col is None:
        return None

    # If multiple rows exist, we’ll take the first non-null pool list
    for v in df[col].dropna().tolist():
        try:
            if isinstance(v, str):
                arr = ast.literal_eval(v)
            else:
                arr = v
            if isinstance(arr, (list, tuple)) and len(arr) > 0:
                return [int(x) for x in arr]
        except Exception:
            continue
    return None

pool_idx = None
if POOL_TASKS_CSV is not None:
    pool_idx = load_pool_idx_from_tasks_csv(POOL_TASKS_CSV)

print("Pool tasks CSV:", POOL_TASKS_CSV if POOL_TASKS_CSV else "(none found)")
print("pool_idx present:", isinstance(pool_idx, list), "| length:", (len(pool_idx) if isinstance(pool_idx, list) else None))

def sub_qubo(Q_full: np.ndarray, idx: list[int]) -> np.ndarray:
    idx = [int(i) for i in idx]
    Qs = Q_full[np.ix_(idx, idx)]
    # stabilize
    return 0.5 * (Qs + Qs.T)

def bitstring_to_vec(s: str) -> np.ndarray:
    return np.array([1 if ch == "1" else 0 for ch in s.strip()], dtype=int)

def qubo_energy(Qmat: np.ndarray, x: np.ndarray) -> float:
    return float(x.T @ Qmat @ x)

# If Q exists but was too large, we can score energies on the pooled sub-Q if pool_idx matches bitstrings.
if Q is not None and pool_idx is not None and len(pool_idx) > 0 and best_global is not None:
    n_pool = len(pool_idx)
    s = best_global["bitstring"]
    if isinstance(s, str) and len(s) == n_pool and set(s) <= {"0", "1"}:
        Q_sub = sub_qubo(Q, pool_idx)
        x = bitstring_to_vec(s)
        e = qubo_energy(Q_sub, x)
        print("Computed pooled (sub-QUBO) energy for best_global:", e)
        # update payload on disk
        best_payload = json.loads(OUT_BEST.read_text())
        best_payload["energy_subqubo"] = float(e)
        best_payload["pool_idx_len"] = int(n_pool)
        best_payload["pool_tasks_csv"] = str(POOL_TASKS_CSV) if POOL_TASKS_CSV else None
        OUT_BEST.write_text(json.dumps(best_payload, indent=2))
        print("Updated:", OUT_BEST)
    else:
        print("Pool-aware scoring skipped: bitstring length != len(pool_idx) or invalid bitstring.")
else:
    print("Pool-aware scoring skipped: missing Q, pool_idx, or best_global.")


Pool tasks CSV: data/results/09c_sv1_tasks_scenario_C.csv
pool_idx present: False | length: None
Pool-aware scoring skipped: missing Q, pool_idx, or best_global.


### What Cell 5 Just Did

- Attempted to load a `pool_idx` list from an SV1 “tasks ledger” CSV (if present).
- If available, computed a **sub-QUBO** energy on the pooled problem (e.g., 34 qubits) so you still get a meaningful energy score even when the full QUBO is larger.
- Updated `data/results/10d_best_from_s3.json` with `energy_subqubo` when possible.


In [9]:
# ============================================================
# Cell 6 — Decode bitstring → selected trials CSV (if we can map indices to IDs)
# ============================================================

# Optional candidates file (scenario candidates) for mapping
CANDIDATE_PATHS = [
    Path("data/processed/scenario_C_candidates.csv"),
    Path("data/processed/scenario_C_candidates_scored.csv"),
    Path("data/results/scenario_C_candidates.csv"),
    Path("data/results/scenario_C_candidates_scored.csv"),
]
CANDIDATES_CSV = first_existing(CANDIDATE_PATHS)

OUT_SELECTED = RESULTS_DIR / "10d_selected_trials_from_best.csv"

def derive_ids_for_index(nct_ids_full: list[str] | None, cand_csv: Path | None):
    """
    Returns an ordered list of IDs aligned to Q indices (best effort).
    Priority:
      1) nct_ids parsed from QUBO JSON
      2) candidates CSV nct_id column
    """
    if nct_ids_full is not None and isinstance(nct_ids_full, list) and len(nct_ids_full) > 0:
        return [str(x) for x in nct_ids_full]

    if cand_csv is not None and cand_csv.exists():
        df = pd.read_csv(cand_csv)
        if "nct_id" in df.columns:
            return [str(x) for x in df["nct_id"].tolist()]
    return None

id_list = derive_ids_for_index(nct_ids, CANDIDATES_CSV)
print("Candidates CSV:", CANDIDATES_CSV if CANDIDATES_CSV else "(none found)")
print("ID list available:", bool(id_list), "| length:", (len(id_list) if id_list else None))

def decode_selected(bitstring: str, ids: list[str], idx_map: list[int] | None):
    """
    Returns selected IDs for:
      - full-Q indexing (idx_map=None) OR
      - pooled indexing (idx_map provided): bit position j maps to full index idx_map[j]
    """
    x = [1 if ch == "1" else 0 for ch in bitstring]
    selected = []
    if idx_map is None:
        for i, xi in enumerate(x):
            if xi == 1 and i < len(ids):
                selected.append(ids[i])
    else:
        for j, xj in enumerate(x):
            if xj == 1:
                full_i = int(idx_map[j])
                if full_i < len(ids):
                    selected.append(ids[full_i])
    return selected

if best_global is None:
    raise RuntimeError("best_global is missing (run Cell 4 first).")

bit = best_global["bitstring"]
idx_map = pool_idx if (pool_idx is not None and len(bit) == len(pool_idx)) else None

if id_list is None:
    print("Skip: cannot write selected trials (no ID mapping found).")
else:
    selected_ids = decode_selected(bit, id_list, idx_map)
    df_out = pd.DataFrame({"nct_id": selected_ids})
    df_out.to_csv(OUT_SELECTED, index=False)
    print("Wrote:", OUT_SELECTED, "| selected_n:", len(df_out))
    display(df_out.head(15))


Candidates CSV: data/processed/scenario_C_candidates.csv
ID list available: True | length: 60
Wrote: data/results/10d_selected_trials_from_best.csv | selected_n: 2


,nct_id
0,NCT05797610
1,NCT06687967


### What Cell 6 Just Did

- Tried to map indices back to real trial IDs using either:
  - ID list embedded in the QUBO JSON, or
  - a scenario candidates CSV containing `nct_id`.
- If the SV1 run used pooling, it used `pool_idx` to map pooled bits back to full candidate indices.
- Wrote `data/results/10d_selected_trials_from_best.csv` when mapping was possible.


In [10]:
# ============================================================
# Cell 7 — Export a comparison-ready single-row summary
# ============================================================

OUT_ONE_ROW = RESULTS_DIR / "10d_best_one_row.csv"

best_payload = json.loads(OUT_BEST.read_text())

row = {
    "braket_bucket": best_payload.get("braket_bucket"),
    "braket_prefix": best_payload.get("braket_prefix"),
    "results_json_key": best_payload.get("results_json_key"),
    "bitstring": best_payload.get("bitstring"),
    "support": best_payload.get("support"),
    "energy_fullqubo": best_payload.get("energy"),
    "energy_subqubo": best_payload.get("energy_subqubo", None),
    "scoring": best_payload.get("scoring"),
    "measured_qubits_len": best_payload.get("measured_qubits_len"),
    "pool_idx_len": best_payload.get("pool_idx_len", None),
}

pd.DataFrame([row]).to_csv(OUT_ONE_ROW, index=False)
print("Wrote:", OUT_ONE_ROW)
display(pd.DataFrame([row]))


Wrote: data/results/10d_best_one_row.csv


,braket_bucket,braket_prefix,results_json_key,bitstring,support,energy_fullqubo,energy_subqubo,scoring,measured_qubits_len,pool_idx_len
0,amazon-braket-us-west-2-581610642254,clinical-trials-data/results/sv1_tasks/scenari...,clinical-trials-data/results/sv1_tasks/scenari...,0000000000001010000000000000000000,2,None,None,frequency,34,None


### What Cell 7 Just Did

- Created a single-row, comparison-friendly CSV with the best decoded SV1 result:
  - includes bitstring, support, and energy (sub-QUBO if available).
- This is ideal for joining into scenario comparison notebooks later.

## Summary

This notebook is now production-safe for Braket SV1 workflows:

- It resumes from S3 and decodes `results.json` without calling `task.result()`.
- It works even when SV1 is run on a pooled subset of the full QUBO.
- It exports:
  - diagnostics (`10d_decode_diagnostics.csv`)
  - per-task bests (`10d_decoded_rows.csv`)
  - a single best artifact (`10d_best_from_s3.json`)
  - optional selected trials (`10d_selected_trials_from_best.csv`)
  - a one-row comparison file (`10d_best_one_row.csv`)
